In [9]:
# ============================================================
# ADVANCED SYNTHETIC LTR DATASET + LAMBDAMART
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

# ============================================================
# CONFIG
# ============================================================

N_QUERIES = 300
DOCS_PER_QUERY = 10
EMBED_DIM = 32

# ============================================================
# VOCAB
# ============================================================

vocab = [
    "ai","machine","learning","deep","neural",
    "transformer","bert","attention","ranking",
    "search","retrieval","embedding","vector",
    "database","language","model","llm","rag",
    "token","context","document","query"
]

# ============================================================
# EMBEDDING SPACE
# ============================================================

word_vectors = {
    w: np.random.normal(0, 1, EMBED_DIM)
    for w in vocab
}

# ============================================================
# HELPERS
# ============================================================

def text_embedding(tokens):
    vecs = [word_vectors[t] for t in tokens]
    return np.mean(vecs, axis=0)

def cosine(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) *
        np.linalg.norm(b) + 1e-8
    )

def jaccard(a, b):
    sa = set(a)
    sb = set(b)

    return len(sa & sb) / (
        len(sa | sb) + 1e-8
    )

# ============================================================
# DATA GENERATION
# ============================================================

rows = []

qid = 0

for _ in range(N_QUERIES):

    query_tokens = list(
        np.random.choice(vocab, 3, replace=False)
    )

    query_emb = text_embedding(query_tokens)

    for doc_id in range(DOCS_PER_QUERY):

        # ----------------------------
        # HARD NEGATIVES
        # ----------------------------

        r = np.random.rand()

        if r < 0.25:
            overlap = 3

        elif r < 0.50:
            overlap = 2

        elif r < 0.75:
            overlap = 1

        else:
            overlap = 0

        shared = list(
            np.random.choice(
                query_tokens,
                overlap,
                replace=False
            )
        )

        remaining = [
            x for x in vocab
            if x not in shared
        ]

        extra = list(
            np.random.choice(
                remaining,
                6 - overlap,
                replace=True
            )
        )

        doc_tokens = shared + extra

        doc_emb = text_embedding(doc_tokens)

        # ==================================================
        # TRUE LATENT RELEVANCE
        # ==================================================

        shared_terms = len(
            set(query_tokens) &
            set(doc_tokens)
        )

        semantic_sim = cosine(
            query_emb,
            doc_emb
        )

        latent_score = (
            2.0 * shared_terms +
            3.0 * semantic_sim +
            np.random.normal(0, 0.25)
        )

        if latent_score < 1:
            label = 0
        elif latent_score < 2:
            label = 1
        elif latent_score < 3:
            label = 2
        elif latent_score < 4:
            label = 3
        else:
            label = 4

        # ==================================================
        # FEATURES
        # ==================================================

        bm25_score = (
            shared_terms +
            np.random.normal(0, 0.2)
        )

        tfidf_cosine = (
            semantic_sim +
            np.random.normal(0, 0.1)
        )

        jac = jaccard(
            query_tokens,
            doc_tokens
        )

        query_len = len(query_tokens)
        doc_len = len(doc_tokens)

        length_ratio = (
            query_len / doc_len
        )

        interaction_1 = (
            bm25_score *
            semantic_sim
        )

        interaction_2 = (
            shared_terms *
            semantic_sim
        )

        rows.append([
            qid,
            label,

            bm25_score,
            tfidf_cosine,
            semantic_sim,

            shared_terms,
            jac,

            query_len,
            doc_len,
            length_ratio,

            interaction_1,
            interaction_2
        ])

    qid += 1

# ============================================================
# DATAFRAME
# ============================================================

cols = [
    "qid",
    "label",

    "bm25",
    "tfidf",

    "semantic",

    "shared_terms",
    "jaccard",

    "query_len",
    "doc_len",
    "length_ratio",

    "interaction1",
    "interaction2"
]

df = pd.DataFrame(rows, columns=cols)

# ============================================================
# SPLIT BY QUERY
# ============================================================

train_qids, test_qids = train_test_split(
    df.qid.unique(),
    test_size=0.2,
    random_state=42
)

train_df = df[
    df.qid.isin(train_qids)
].copy()

test_df = df[
    df.qid.isin(test_qids)
].copy()

# ============================================================
# FEATURES
# ============================================================

FEATURES = [
    "bm25",
    "tfidf",
    "semantic",

    "shared_terms",
    "jaccard",

    "query_len",
    "doc_len",
    "length_ratio",

    "interaction1",
    "interaction2"
]

X_train = train_df[FEATURES]
y_train = train_df["label"]

X_test = test_df[FEATURES]
y_test = test_df["label"]

_, group_train = np.unique(
    train_df.qid,
    return_counts=True
)

_, group_test = np.unique(
    test_df.qid,
    return_counts=True
)

# ============================================================
# LAMBDAMART
# ============================================================

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",

    verbosity=-1,

    n_estimators=1000,
    learning_rate=0.03,

    num_leaves=255,

    feature_fraction=0.8,

    bagging_fraction=0.8,
    bagging_freq=5,

    lambda_l1=0.1,
    lambda_l2=1.0,

    random_state=42
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)

# ============================================================
# NDCG
# ============================================================

preds = ranker.predict(X_test)

scores = []

start = 0

for g in group_test:

    end = start + g

    scores.append(
        ndcg_score(
            [y_test.iloc[start:end]],
            [preds[start:end]],
            k=10
        )
    )

    start = end

print(
    "\nNDCG@10:",
    round(np.mean(scores), 4)
)

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance":
        ranker.feature_importances_
})

importance = (
    importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFEATURE IMPORTANCE\n")
print(importance)

# ============================================================
# ESCOLHE UMA QUERY DE TESTE
# ============================================================

sample_qid = test_df.qid.iloc[0]

q = (
    test_df[
        test_df.qid == sample_qid
    ]
    .copy()
)

# ============================================================
# SCORES DO MODELO
# ============================================================

q["pred_score"] = ranker.predict(
    q[FEATURES]
)

# ============================================================
# RANKING PREVISTO
# ============================================================

q["pred_rank"] = (
    q["pred_score"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

# ============================================================
# RANKING REAL
# ============================================================

q["true_rank"] = (
    q["label"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

# ============================================================
# ERRO DE POSIÇÃO
# ============================================================

q["rank_error"] = (
    q["pred_rank"]
    -
    q["true_rank"]
)

# ============================================================
# ORDENA PELO RANK PREVISTO
# ============================================================

ranking_table = (
    q[
        [
            "label",
            "pred_score",

            "true_rank",
            "pred_rank",

            "rank_error",

            "bm25",
            "tfidf",
            "semantic",

            "shared_terms",
            "jaccard"
        ]
    ]
    .sort_values(
        "pred_rank"
    )
    .reset_index(drop=True)
)

print("\nRANKING REAL VS PREVISTO\n")
print(ranking_table)

# ============================================================
# SPEARMAN
# ============================================================

from scipy.stats import spearmanr

corr, pvalue = spearmanr(
    ranking_table["true_rank"],
    ranking_table["pred_rank"]
)

print(
    "\nSpearman Rank Correlation:",
    round(corr, 4)
)

# ============================================================
# ERRO MÉDIO ABSOLUTO DE RANK
# ============================================================

mae_rank = np.mean(
    np.abs(
        ranking_table["rank_error"]
    )
)

print(
    "Mean Absolute Rank Error:",
    round(mae_rank, 4)
)

# ============================================================
# TOP ACERTOS
# ============================================================

perfect = ranking_table[
    ranking_table["rank_error"] == 0
]

print(
    "\nDocumentos com posição correta:",
    len(perfect),
    "/",
    len(ranking_table)
)

# ============================================================
# MAIORES ERROS
# ============================================================

worst = (
    ranking_table
    .reindex(
        ranking_table["rank_error"]
        .abs()
        .sort_values(
            ascending=False
        )
        .index
    )
)

print("\nMAIORES ERROS\n")
print(
    worst.head(10)
)

# ============================================================
# MATRIZ RESUMO
# ============================================================

summary = pd.DataFrame({
    "Metric": [
        "NDCG@10",
        "Spearman",
        "Mean Abs Rank Error"
    ],
    "Value": [
        np.mean(scores),
        corr,
        mae_rank
    ]
})

print("\nRESUMO\n")
print(summary)




NDCG@10: 0.9997

FEATURE IMPORTANCE

        feature  importance
0  interaction2         904
1          bm25         751
2      semantic         562
3  shared_terms         454
4         tfidf         378
5  interaction1         340
6       jaccard         164
7     query_len           0
8  length_ratio           0
9       doc_len           0

RANKING REAL VS PREVISTO

   label  pred_score  true_rank  pred_rank  rank_error      bm25     tfidf  \
0      4    5.015089          1          1           0  2.405671  0.625931   
1      4    5.015089          1          1           0  1.788272  0.556827   
2      4    5.015089          1          1           0  1.949463  0.603472   
3      4    5.015089          1          1           0  3.034194  0.428705   
4      4    5.007918          1          2           1  1.980332  0.256201   
5      4    5.007918          1          2           1  1.907410  0.255357   
6      4    5.006373          1          3           2  2.144293  0.375939   
7  